# Lecture: VAE on Real Images — Generating Faces with CelebA

So far the VAE was demonstrated on MNIST and Fashion-MNIST: small, grayscale,
28x28 images. Those datasets make the *mechanics* easy to see (especially the
2D latent space), but they hide how powerful a VAE becomes on **real, natural
images**.

This notebook applies the **same VAE principle** to [**CelebA**](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html),
a dataset of ~200,000 celebrity face photographs. We move to **RGB 64x64**
images and a much larger latent space (`latent_dim=256`).

The goal here is **not** to train from scratch — that needs a GPU and ~1-1.5 h
on a free Colab T4. Instead, this notebook is **load-only**: we load a
pre-trained model and explore what a face VAE can actually do:

1. **Reconstruction** — compress a face to 256 numbers and decode it back.
2. **Sampling from the prior** — invent entirely new faces of people who do not exist.
3. **Latent interpolation** — morph smoothly from one face into another.
4. **Attribute arithmetic** — find the "smile" or "glasses" direction in latent
   space and add it to any face.

To reproduce the weights yourself, the standalone training script
`train_vae_celeba.py` is provided for running on a GPU server (see the
final section).

## Setup (Google Colab only)

Run the following cell **only on Google Colab** to clone the repository and
copy the required `VAE_CelebA.py` module into the working directory. If you
work locally, skip it.

In [ ]:
# NOTE: clones the 'vae-celeba' branch for testing before merge.
# After merge into main, drop the '-b vae-celeba' flag.
!git clone -b vae-celeba https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C2-Autoencoders/VAE_CelebA.py ./

## Loading the Pre-trained Model

The pre-trained weights (`vae_celeba.pth`, latent_dim=256) are committed to the
repository under `C2-Autoencoders/models/`. The cell below picks the correct
path automatically:

- **locally**: `models/vae_celeba.pth` (the notebook runs from this folder), or
- **on Colab**: `AIBIP/C2-Autoencoders/models/vae_celeba.pth` (after the clone
  in the setup cell).

No download is required.

In [ ]:
import os
import torch
from VAE_CelebA import VAE

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

LATENT_DIM = 256

# The pre-trained weights are committed to the repository under
# C2-Autoencoders/models/. Locally the notebook runs from that folder, so the
# relative path "models/vae_celeba.pth" works directly. On Colab the repo is
# cloned into ./AIBIP, so we also check the cloned location.
CANDIDATE_PATHS = [
    "models/vae_celeba.pth",                           # local run
    "AIBIP/C2-Autoencoders/models/vae_celeba.pth",     # Colab after git clone
]
weights_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), CANDIDATE_PATHS[0])

model = VAE(latent_dim=LATENT_DIM).to(device)
model.load_model(path=weights_path, device=device)
model.eval()

## A Few Real Faces to Work With

For the demos we need a handful of real CelebA faces. To keep the notebook
runnable on Colab **without downloading the full CelebA dataset**, a small set
of pre-processed example faces is shipped with the repository in
`celeba_demo_assets/sample_faces.pt` (already center-cropped and resized to 64x64). They were
exported with `export_celeba_demo_assets.py` (see the final section).

In [ ]:
import os

# Pre-processed example faces: a dict with "images" (N, 3, 64, 64) in [0, 1].
CANDIDATE_PATHS = [
    "celeba_demo_assets/sample_faces.pt",                           # local run
    "AIBIP/C2-Autoencoders/celeba_demo_assets/sample_faces.pt",     # Colab after git clone
]
faces_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), CANDIDATE_PATHS[0])

faces = torch.load(faces_path, map_location=device)
images = faces["images"].to(device)
print("loaded", images.size(0), "example faces:", tuple(images.shape))

## 1) Reconstruction

The VAE compresses each 64x64x3 = 12,288-dimensional image into a 256-number
latent code and decodes it back. Reconstructions are recognisably the same
person but slightly smoothed — the hallmark "blurriness" of VAEs, caused by the
Gaussian likelihood and the averaging effect of the KL regulariser.

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    recon, _, _ = model(images)

n = 8
fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
for i in range(n):
    axes[0, i].imshow(images[i].permute(1, 2, 0).cpu().numpy())
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].permute(1, 2, 0).cpu().numpy())
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("original",       rotation=0, ha="right", labelpad=40)
axes[1, 0].set_ylabel("reconstruction", rotation=0, ha="right", labelpad=40)
plt.tight_layout()
plt.show()

## 2) Sampling New Faces from the Prior

This is the capability a plain autoencoder lacks. Because the latent space is
regularised towards `N(0, I)`, we can simply draw random vectors `z ~ N(0, I)`
and decode them into **new faces of people who do not exist**.

In [ ]:
with torch.no_grad():
    samples = model.sample(16, device=device)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].permute(1, 2, 0).cpu().numpy())
    ax.axis("off")
plt.suptitle("Faces sampled from the prior  z ~ N(0, I)")
plt.tight_layout()
plt.show()

## 3) Latent-Space Interpolation (Face Morphing)

Encoding two faces to their posterior means and decoding the straight line
between them produces a **smooth morph** — every intermediate point is a valid
face. This demonstrates that the latent space is continuous and semantically
meaningful, not just a lookup table of memorised images.

In [ ]:
steps = 10
morph = model.interpolate(images[0], images[1], steps=steps)

fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2.2))
for i in range(steps):
    axes[i].imshow(morph[i].permute(1, 2, 0).cpu().numpy())
    axes[i].axis("off")
plt.suptitle("Interpolation from face A (left) to face B (right)")
plt.tight_layout()
plt.show()

## 4) Attribute Arithmetic

CelebA ships 40 binary attribute labels per image (e.g. *Smiling*, *Eyeglasses*,
*Male*, *Blond_Hair*). The latent **direction** of an attribute is the
difference between the *mean latent code of images that have it* and the *mean
latent code of images that lack it*:

$$\mathbf{v}_{\text{attr}} = \overline{\mathbf{z}}_{\text{has attr}} - \overline{\mathbf{z}}_{\text{no attr}}$$

Estimating these directions requires encoding thousands of *labelled* images,
so they are **pre-computed** (with `export_celeba_demo_assets.py`) and shipped
in `celeba_demo_assets/attribute_vectors.pt`. Adding $\alpha \cdot \mathbf{v}_{\text{attr}}$ to a
face's latent code and decoding turns the attribute up or down continuously.

In [ ]:
# Pre-computed latent attribute directions: name -> (latent_dim,) tensor.
CANDIDATE_PATHS = [
    "celeba_demo_assets/attribute_vectors.pt",                           # local run
    "AIBIP/C2-Autoencoders/celeba_demo_assets/attribute_vectors.pt",     # Colab after git clone
]
vec_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), CANDIDATE_PATHS[0])

attribute_vectors = {k: v.to(device) for k, v in torch.load(vec_path, map_location=device).items()}
print("available attribute directions:", list(attribute_vectors.keys()))

smile_vec = attribute_vectors["Smiling"]
print("'Smiling' direction, norm =", round(smile_vec.norm().item(), 3))

In [ ]:
# Apply the attribute vector to one face across a range of strengths.
face = images[2:3]
with torch.no_grad():
    z = model.encode(face)
    alphas = torch.linspace(-3, 3, 7, device=device)
    edited = torch.cat([model.decode(z + a * smile_vec) for a in alphas])

fig, axes = plt.subplots(1, len(alphas), figsize=(2 * len(alphas), 2.4))
for i, a in enumerate(alphas):
    axes[i].imshow(edited[i].permute(1, 2, 0).cpu().numpy())
    axes[i].set_title(f"a={a:.0f}")
    axes[i].axis("off")
plt.suptitle("Adding the 'Smiling' direction (left = less, right = more)")
plt.tight_layout()
plt.show()

---
## Reproducing the Weights

This notebook is **load-only** — the checkpoint shipped in
`C2-Autoencoders/models/vae_celeba.pth` was produced with the standalone script
[`train_vae_celeba.py`](train_vae_celeba.py) on a GPU server (training on CPU is
impractical).

The CelebA images were obtained from a local folder to avoid the rate-limited
Google Drive mirror that `torchvision.datasets.CelebA` uses (e.g. the aligned
images from the Kaggle CelebA mirror):

```bash
python train_vae_celeba.py --data-dir <folder-with-images>     --epochs 25 --batch-size 128 --lr 1e-3
```

This writes `models/vae_celeba.pth`. See the script's `--help` for all options
(`--beta`, `--latent-dim`, ...).